<a href="https://colab.research.google.com/github/iamabdelhadi02/Anthropic-Academy/blob/main/001_prompt_evals_grader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: Load env variables and create client
# ============================================================

# --- ORIGINAL CODE (Anthropic SDK) ---
# from dotenv import load_dotenv
# from anthropic import Anthropic
#
# load_dotenv()
#
# client = Anthropic()
# model = "claude-haiku-4-5"

# --- MODIFIED CODE (OpenRouter via OpenAI SDK) ---
# WHY: OpenRouter exposes an OpenAI-compatible API, so we use the `openai` package
# instead of `anthropic`. We point `base_url` to OpenRouter's endpoint and use
# an OpenRouter API key instead of an Anthropic one.
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",  # OpenRouter's OpenAI-compatible endpoint
    api_key=os.getenv("OPENROUTER_API_KEY"),   # Free key from openrouter.ai
)

# WHY: open router looks for the freely available model so you won't hit the  usage limit
# following and JSON output — best free alternative on OpenRouter for this exercise.
model = "openrouter/free"

In [ ]:
# ============================================================
# CELL 2: Helper functions
# ============================================================

# add_user_message and add_assistant_message are UNCHANGED — the message
# format {"role": ..., "content": ...} is the same in both APIs.

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


# --- ORIGINAL chat() FUNCTION (Anthropic SDK) ---
# def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
#     params = {
#         "model": model,
#         "max_tokens": 1000,
#         "messages": messages,
#         "temperature": temperature,
#         "stop_sequences": stop_sequences,  # Anthropic param name
#     }
#
#     if system:
#         params["system"] = system  # Anthropic accepts system as a top-level param
#
#     message = client.messages.create(**params)  # Anthropic SDK method
#     return message.content[0].text              # Anthropic response structure

# --- MODIFIED chat() FUNCTION (OpenAI SDK via OpenRouter) ---
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    # WHY: OpenAI API uses "system" as a message role inside the messages list,
    # NOT as a separate top-level parameter like Anthropic does.
    api_messages = []
    if system:
        api_messages.append({"role": "system", "content": system})
    api_messages.extend(messages)

    # WHY: OpenAI API uses `client.chat.completions.create()` instead of
    # Anthropic's `client.messages.create()`. Also the stop param is called
    # `stop` (not `stop_sequences`).
    response = client.chat.completions.create(
        model=model,
        max_tokens=1000,
        messages=api_messages,
        temperature=temperature,
        stop=stop_sequences if stop_sequences else None,  # Renamed param
    )

    # WHY: OpenAI response structure is response.choices[0].message.content
    # instead of Anthropic's message.content[0].text
    return response.choices[0].message.content

In [ ]:
# ============================================================
# CELL 2.5: JSON cleaning utility (NEW — not in original)
# ============================================================

# WHY: The original Anthropic notebook uses "assistant prefilling" — it injects
# a partial assistant message like '```json' to force the model to continue
# generating JSON, then uses stop_sequences=['```'] to cut it off cleanly.
# The OpenAI API does NOT support assistant prefilling the same way.
# Instead, we instruct the model to output raw JSON and then strip any
# markdown fences the model might still add.

def clean_json_response(text):
    """Strip markdown code fences from a model response to get raw JSON."""
    cleaned = text.strip()
    # Remove opening fence like ```json or ```
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned[3:]
    # Remove closing fence
    if cleaned.endswith("```"):
        cleaned = cleaned.rsplit("```", 1)[0]
    cleaned = cleaned.strip()
    # Handle case where 'json' label remains after fence removal
    if cleaned.startswith("json"):
        cleaned = cleaned[4:].strip()
    return cleaned

In [ ]:
# ============================================================
# CELL 3: Function to generate a new dataset
# ============================================================
import json


# --- ORIGINAL generate_dataset() (Anthropic SDK with prefilling) ---
# def generate_dataset():
#     prompt = """
# Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
# that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
# each representing task that requires Python, JSON, or a Regex to complete.
#
# Example output:
# ```json
# [
#     {
#         "task": "Description of task",
#         "format": "json" or "python" or "regex"
#     },
#     ...additional
# ]
# ```
#
# * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
# * Focus on tasks that do not require writing much code
#
# Please generate 3 objects.
# """
#
#     messages = []
#     add_user_message(messages, prompt)
#     # WHY THIS WAS REMOVED: "Assistant prefilling" — injecting a partial assistant
#     # response to steer the model. This is an Anthropic-specific technique.
#     # OpenAI API doesn't support it; we use explicit instructions instead.
#     add_assistant_message(messages, "```json")
#     text = chat(messages, stop_sequences=["```"])
#     return json.loads(text)

# --- MODIFIED generate_dataset() (OpenRouter / OpenAI SDK) ---
def generate_dataset():
    # WHY: Instead of prefilling + stop_sequences, we add an explicit instruction
    # telling the model to respond with ONLY raw JSON. This achieves the same
    # result (clean JSON output) without relying on Anthropic-specific features.
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.

IMPORTANT: Respond with ONLY the raw JSON array. No markdown fences, no explanation, just the JSON.
"""

    messages = []
    add_user_message(messages, prompt)
    # NOTE: No more add_assistant_message("```json") or stop_sequences
    text = chat(messages)
    return json.loads(clean_json_response(text))

In [ ]:
# ============================================================
# CELL 4: Generate the dataset and write it to 'dataset.json'
# ============================================================
# UNCHANGED — this cell just calls generate_dataset() and saves the result.
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
# ============================================================
# CELL 5: Function to grade a test case + output using a model
# ============================================================

# --- ORIGINAL grade_by_model() (Anthropic SDK with prefilling) ---
# def grade_by_model(test_case, output):
#     eval_prompt = f"""
# You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
#
# Original Task:
# <task>
# {test_case["task"]}
# </task>
#
# Solution to Evaluate:
# <solution>
# {output}
# </solution>
#
# Output Format
# Provide your evaluation as a structured JSON object with the following fields, in this specific order:
# - "strengths": An array of 1-3 key strengths
# - "weaknesses": An array of 1-3 key areas for improvement
# - "reasoning": A concise explanation of your overall assessment
# - "score": A number between 1-10
#
# Respond with JSON. Keep your response concise and direct.
# Example response shape:
# {{
#     "strengths": string[],
#     "weaknesses": string[],
#     "reasoning": string,
#     "score": number
# }}
#     """
#
#     messages = []
#     add_user_message(messages, eval_prompt)
#     add_assistant_message(messages, "```json")  # Prefilling trick
#     eval_text = chat(messages, stop_sequences=["```"])
#     return json.loads(eval_text)

# --- MODIFIED grade_by_model() (OpenRouter / OpenAI SDK) ---
def grade_by_model(test_case, output):
    # WHY: Same change as generate_dataset() — replaced assistant prefilling
    # with an explicit "respond with ONLY raw JSON" instruction.
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

IMPORTANT: Respond with ONLY the raw JSON object. No markdown fences, no explanation, just the JSON.
Example response shape:
{{
    "strengths": ["string", ...],
    "weaknesses": ["string", ...],
    "reasoning": "string",
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    # NOTE: No more add_assistant_message("```json") or stop_sequences
    eval_text = chat(messages)
    return json.loads(clean_json_response(eval_text))

In [ ]:
# ============================================================
# CELL 6: Passes a test case into the model
# ============================================================

# --- ORIGINAL ---
# # Passes a test case into Claude
# def run_prompt(test_case):
#     ...

# UNCHANGED — this cell works identically with both APIs since
# it just calls chat() which we already adapted above.
# Only the comment was updated ("Claude" -> "the model").

def run_prompt(test_case):
    """Passes a test case into the model (was Claude, now Gemma 4 via OpenRouter)"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
import re
import ast
import json
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0



def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
# ============================================================
# CELL 7: Execute a single test case and grade the output
# ============================================================
# UNCHANGED — no API-specific code here, just orchestration.

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output,test_case)
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [ ]:
# ============================================================
# CELL 8: Run full evaluation
# ============================================================
# UNCHANGED — no API-specific code here, just orchestration.

from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
# ============================================================
# CELL 9: Execute everything
# ============================================================
# UNCHANGED — loads dataset and runs the eval pipeline.

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.833333333333334


In [ ]:
# ============================================================
# CELL 10: Print results
# ============================================================
# UNCHANGED

print(json.dumps(results, indent=2))

[
  {
    "output": "import boto3\n\ndef get_s3_log_keys(bucket_name):\n    s3 = boto3.client('s3')\n    paginator = s3.get_paginator('list_objects_v2')\n    log_keys = []\n    for page in paginator.paginate(Bucket=bucket_name):\n        if 'Contents' in page:\n            for obj in page['Contents']:\n                key = obj['Key']\n                if key.endswith('.log'):\n                    log_keys.append(key)\n    return log_keys",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and returns a list of all object keys that have the '.log' extension.",
      "format": "python"
    },
    "score": 8.5,
    "reasoning": "The implementation correctly retrieves all '.log' keys using pagination and filtering, making it functional for many scenarios. However, it omits robust error handling, configurability, and documentation, which limits reliability in production environments."
  },
  {
    "output": "{\n  \"Version\": \"2012-10-17\",\n  \"Stateme